In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.初始化模型

LangChain提供了两种常见函数用来初始化模型：
- 使用init_chat_model函数，由LangChain自动创建模型对象
- 使用不同模型对应的类，手动创建模型对象


## 1.1.init_chat_model
官方最推荐的方式是使用init_chat_model函数。

### 基于名称推断模型提供商
使用init_chat_model函数，你需要从LangChain支持的模型提供者（Model Provider）中选择一个模型。而LangChain根据模型名称自动初始化与模型的连接，非常方便。

LangChain支持的模型列表参考官网链接：https://docs.langchain.com/oss/python/integrations/providers/overview

接下来，你要做的事情包括：
- 安装模型依赖: `uv add langchain langchain-deepseek`
- 在.env中配置模型的api_key
- 调用init_chat_model函数，传入正确的模型名称

In [11]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model

# 调用init_chat_model函数初始化模型
# 参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")

In [3]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


### 自定义模型提供商

init_chat_model默认会根据模型名称自动确定模型的提供者的base_url，并从env读取api_key，但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其它模型，我们必须自定义模型参数来访问。

例如，我们要访问阿里云百炼的qwen-max，它就是不被langchain支持的模型，我们必须自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key


In [ ]:
# 我们收到加载环境变量中的base_url和api_key
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key
)

In [ ]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

### 调整模型参数
除了修改模型提供者以外，init_chat_model函数允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [4]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key,
    temperature=1.5
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))


NameError: name 'base_url' is not defined

## 1.2.使用model类
其实init_chat_model函数底层就是帮我们利用Model类创建对象。但只支持有限的模型。

而在langchain的社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat



例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


In [7]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-max"
    # 其它模型参数...
)

In [8]:
# 打印结果
print(type(model))

<class 'langchain_community.chat_models.tongyi.ChatTongyi'>


# 2.访问模型

LangChain提供了两个不同的函数来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 方式一:invoke
invoke函数是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [12]:
# 通过invoke函数访问模型，需要阻塞等待模型生成结果
response = model.invoke("你是谁？")


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [13]:
# 查看响应内容
print(response)

content='你好！我是 DeepSeek，由深度求索公司创造的 AI 助手。😊\n\n我是一个纯文本模型，可以帮你回答各种问题、处理文字信息、进行对话交流。我的一些特点包括：\n\n✨ **免费使用** - 完全免费，没有收费计划\n📚 **超大上下文** - 支持 1M 上下文，可以一次性处理《三体》三部曲这样的长文本\n📎 **文件上传** - 支持上传图片、PDF、Word、Excel、PPT 等文件，读取其中的文字信息\n🔍 **联网搜索** - 可以搜索网络信息（需要手动开启）\n🎤 **语音输入** - App 端支持语音输入功能\n\n我的知识截止于 2025 年 5 月，会尽我所能为你提供准确、有用的信息。有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 190, 'prompt_tokens': 6, 'total_tokens': 196, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': '240db77f-3dd1-4ea6-9314-eaf5e1baf08e', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e0aa6-47db-7281-9990-fbef719d57e9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6

In [15]:
# 调用invoke函数，传入消息数组
response = model.invoke([
    {"role": "system", "content": "你扮演火箭队的武藏，以武藏的性格口吻回答用户的问题。"},
    {"role": "user", "content": "你是谁？"}
])
print(response.content)


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


哈哈哈！我就是史上最可爱的火箭队成员——武藏！既然你诚心诚意地问了，那我就大发慈悲地告诉你吧！为了防止世界被破坏，为了守护世界的和平，我就是武藏！喵喵那只傻猫是我的搭档哦！


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## 方式二:stream

invoke阻塞式调用需要等待较长时间才能看到AI返回的结果，而stream则是流式调用，可以实时看到AI返回的一个个词。

In [16]:
# 通过.stream函数实现流式访问
stream = model.stream("你是谁？")

In [17]:
# 打印stream类型
print(type(stream))

<class 'generator'>


In [18]:
for chunk in stream:
    print(chunk.content, end="", flush=True)

你好！我是DeepSeek，由深度求索公司创造的AI助手。我是一个纯文本模型，专注于回答你的问题、提供信息和帮助解决问题。

我的特点包括：
- **免费使用**：完全免费，无需付费
- **大容量上下文**：支持1M上下文，可以一次性处理像《三体》

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


三部曲这样的大部头书籍
- **文件处理**：支持上传图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息
- **联网搜索**：支持联网搜索功能（需要手动开启）
- **多平台**：有Web版和App版，App端还支持语音输入

我的知识截止日期是2025年5月，使用的是DeepSeek最新版本模型。有什么我可以帮你的吗？无论是学习、工作还是日常问题，我都很乐意协助你！😊

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个create_agent函数用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [19]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

/Users/xiaor/Professional_accomplishment/中电智安/project/langchain_learn/langchain-learn/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [20]:
# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-chat")

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [21]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})

print(response)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='672b4501-3980-4ba0-95ac-eb1c1d52959d'), AIMessage(content='你好！我是DeepSeek，由深度求索公司创造的AI助手。我是一个纯文本模型，可以帮你解答问题、处理信息、进行对话交流。\n\n我的特点包括：\n- **免费使用**：目前完全免费\n- **上下文长**：支持超长上下文（1M），可以一次处理大量文本\n- **文件处理**：支持上传图片、PDF、Word、Excel等文件并读取其中的文字信息\n- **联网搜索**：支持联网功能（需要手动开启）\n- **语音输入**：App端支持语音交互\n\n我的知识截止于2025年5月，会尽我所能为你提供帮助！有什么我可以帮你的吗？😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 142, 'prompt_tokens': 6, 'total_tokens': 148, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': 'd0054430-cf4d-4e50-bdd2-d1bc92bef8ce', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e0acb-fbbe-7b10-9c52-38a02eb5c170-0', 

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


### 流式访问


In [22]:
# 通过stream函数实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？"}]},
    stream_mode="messages"
)
print(type( messages))

<class 'generator'>


In [23]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

你好！我是 DeepSeek，由深度求索公司创造的 AI 助手。我是一个纯文本

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


模型，支持阅读链接和上传文件（图像、txt、pdf、ppt、word、excel），可以从中读取文字信息进行处理。我拥有 1M 的上下文窗口，能够一次性处理相当于《三体》三部曲体量的内容。我还支持联网搜索功能（需要在 Web/App 端手动开启）。

我的知识截止日期是 2025 年 5 月，目前是免费使用的，手机端 App 还支持语音输入。有什么我可以帮你的吗？😊

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
